# Atlas-400F (notional cargo jet) — requirements-first program + report

Walkthrough: **Level-1 requirements** are written with ThunderGraph’s normal APIs (`requirement`, `requirement_input`, **`requirement_attribute`** for derived quantities on the requirement, `requirement_accept_expr`, `allocate`, `references`) in `examples/commercial_aircraft/program/`. The program root then composes an **`Aircraft`**, roll-ups, **two illustrative external tools**, evaluation, **report extract**, and an ASCII snapshot.

**Import path:** the example package is `thundergraph-model/examples/commercial_aircraft/`. The next cell adds the library root (for `tg_model`) and `examples/` to Python’s import path. After you change Python files in the example, restart the kernel and run from the top.


## What this demo is proving (two separate checks — read the report banner)

1. **Mission planning desk (illustrative):** a small external tool scales a **baseline maximum range** with payload and subtracts the **requested design range**, producing `mission_range_margin_m`. The constraint `mission_range_margin_non_negative` only checks that desk output. It is **not** a certification or compliance result.
2. **Declared envelope:** separate program constraints compare the **scenario** payload and range to **rolled-up masses** and **parameters** you set (for example maximum design range, maximum takeoff weight, maximum zero-fuel weight). This path does **not** use the desk’s physics.

A **PASS** in the summary means both checks passed for the chosen numbers — **not** that one unified physics model closed the mission.

**Level-1 table:** the **verification** column in the printed report is a short label for how each requirement is meant to be read: executable acceptance (mission closure with `allocate` inputs plus **`requirement_attribute`** margins on the requirement), shown by constraints, or context and citations only (regulatory framing — not a certification claim).

**Mission closure detail:** the mission block is split into **two** atomic Level-1 requirements (payload vs range, INCOSE-style). Each has its own **`requirement_input`** wiring, a **`requirement_attribute`** margin (`payload_margin_kg` or `range_margin_m`), and a **`requirement_accept_expr`** on that margin. After `instantiate`, those derived slots appear on **`ConfiguredModel.requirement_value_slots`** (see the short showcase cell below the path setup).

**Scope:** this notebook is a ThunderGraph API walkthrough (composition, `parameter_ref`, external compute in two places). It is not a full multi-discipline aircraft model.


## Composition (high level)

| Layer | Role |
|-------|------|
| `CargoJetProgram` | Root program: scenario parameters, citations, Level-1 requirement blocks, mission desk → `mission_range_margin_m`, envelope constraints |
| `Aircraft` | Vehicle part: assembly tree, operating empty weight roll-up, modeled maximum payload |
| Major assemblies | Stub dry masses + wing assembly structural intensity (second external tool) |
| `requirement_attribute` (mission closure) | One derived margin per atomic mission requirement (payload block vs range block); materialized on `instantiate` as entries in `cm.requirement_value_slots` |


In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def _thundergraph_pkg_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(24):
        if (p / "tg_model" / "__init__.py").is_file():
            return p
        nested = p / "thundergraph-model"
        if (nested / "tg_model" / "__init__.py").is_file():
            return nested
        if p.parent == p:
            break
        p = p.parent
    return start.resolve()


_cwd = Path.cwd().resolve()
_pkg_root = _thundergraph_pkg_root(_cwd)
_examples = _pkg_root / "examples"
for _p in (_pkg_root, _examples):
    _s = str(_p)
    if _s in sys.path:
        sys.path.remove(_s)
    sys.path.insert(0, _s)



## Level-1 mission closure: `requirement_input` + `requirement_attribute`

In `program/l1_requirement_blocks.py`, **mission closure** is two requirements: **payload** (`req_cargo_design_mission_payload_closure`) and **range** (`req_cargo_design_mission_range_closure`). `CargoJetProgram` calls **`allocate`** twice on the aircraft part—each requirement only receives the inputs it needs. Each defines one **`requirement_attribute`** margin (envelope minus scenario) and a **`requirement_accept_expr`** that the margin is non-negative.

Run the next cell after `instantiate` to print the derived slot paths (also on `cm.requirement_value_slots`).

In [2]:
from commercial_aircraft import CargoJetProgram, reset_commercial_aircraft_types

reset_commercial_aircraft_types()
_cm = CargoJetProgram.instantiate()
print("requirement_attribute slots (derived on the requirement; see l1_requirement_blocks.L1MissionRequirements):")
for _slot in _cm.requirement_value_slots:
    print(" ", _slot.path_string, f"({_slot.kind})")

requirement_attribute slots (derived on the requirement; see l1_requirement_blocks.L1MissionRequirements):
  CargoJetProgram.l1.mission.req_cargo_design_mission_payload_closure.payload_margin_kg (attribute)
  CargoJetProgram.l1.mission.req_cargo_design_mission_range_closure.range_margin_m (attribute)


## Nominal scenario (inside envelope, desk has margin)

Illustrative weight book: **140 t** operating empty weight, **240 t** maximum zero-fuel weight → **100 t** structural payload cap; **95 t** scenario payload. **8,000 km** design range versus **9,000 km** modeled maximum range and a **10,000 km** desk baseline so the illustrative margin stays positive.

**Run path:** **`ConfiguredModel.evaluate`** with **`ValueSlot`** keys (lazy compile; default validation). **`extract_cargo_jet_evaluation_report(cm, result)`** only needs the **`RunResult`**; pass **`ctx=…`** to that helper if you want optional **external-tool provenance** and **slot-state** counts (demo extras, not on **`RunResult`**). An optional **advanced** cell below shows `compile_graph` + `Evaluator`.

Inputs that move the verdict first: **`mission_desk_baseline_max_range_m`** (mission desk), **`scenario_design_range_m`**, **`modeled_max_design_range_m`** (envelope), and assembly dry masses (roll-up / maximum takeoff weight closure).


In [3]:
from unitflow import Quantity
from unitflow.catalogs.si import kg, m

from commercial_aircraft import CargoJetProgram, reset_commercial_aircraft_types
from commercial_aircraft.reporting import extract_cargo_jet_evaluation_report, format_cargo_jet_report


def run_case(label: str, make_inputs) -> None:
    reset_commercial_aircraft_types()
    cm = CargoJetProgram.instantiate()
    ac = cm.aircraft
    inputs = make_inputs(cm, ac)
    result = cm.evaluate(inputs=inputs)
    report = extract_cargo_jet_evaluation_report(cm, result)
    print(f"\n{'=' * 72}\nCASE: {label}\n{'=' * 72}")
    print(format_cargo_jet_report(report))
    print(f"\nEvaluator passed: {result.passed}")
    if result.failures:
        for line in result.failures:
            print("  failure:", line)


def nominal_inputs(cm, ac):
    return {
        cm.scenario_payload_mass_kg: Quantity(95_000, kg),
        cm.scenario_design_range_m: Quantity(8_000_000, m),
        cm.mission_desk_baseline_max_range_m: Quantity(10_000_000, m),
        ac.modeled_max_design_range_m: Quantity(9_000_000, m),
        ac.notional_mzfw_kg: Quantity(240_000, kg),
        ac.notional_mtow_kg: Quantity(280_000, kg),
        ac.notional_trip_fuel_kg: Quantity(40_000, kg),
        ac.fuselage.dry_mass_kg: Quantity(45_000, kg),
        ac.wing.dry_mass_kg: Quantity(32_000, kg),
        ac.wing.notional_wing_span_m: Quantity(64, m),
        ac.empennage.dry_mass_kg: Quantity(8_000, kg),
        ac.landing_gear.dry_mass_kg: Quantity(12_000, kg),
        ac.propulsion_installation.dry_mass_kg: Quantity(28_000, kg),
        ac.aircraft_systems.dry_mass_kg: Quantity(15_000, kg),
    }


run_case("nominal (expect PASS)", nominal_inputs)



CASE: nominal (expect PASS)
Atlas-400F (notional) — evaluation snapshot
Verdict (desk margin + declared-envelope constraints + other checks): PASS
Thesis (read this once)
The demo uses two independent ideas: (1) **Mission desk** — a toy
external scaling model produces `mission_range_margin_m` and
`mission_range_margin_non_negative` checks it. (2) **Declared envelope**
— scenario payload/range are compared to rolled-up masses and parameters
(`notional_takeoff_mass_closure`, payload/range vs envelope). Passing
both is not a single physics closure; it is stitched verification for
teaching.
Mission desk track:
  Margin (raw): 2077236.6898835376 m
  Margin (rounded): ~2,077.2 km
  Margin ≥ 0: True
Declared envelope track (maximum takeoff weight / payload / range vs parameters):
  Envelope constraints all passed: True
  Evaluator completed without engine failures: True

Scenario inputs
quantity                          | value      | human (length)            
------------------------------

## Advanced (optional): explicit `compile_graph` + `Evaluator`

The same run can be written with the low-level pipeline: `compile_graph` → `validate_graph` → `Evaluator` with `inputs` keyed by `stable_id` strings. Use this when you need `evaluate_async`, custom graph inspection, or to match older examples. The default **`ConfiguredModel.evaluate`** path above is equivalent for synchronous runs.


In [4]:
# Equivalent explicit pipeline for the nominal inputs (same outcome as `evaluate` above).
from tg_model.execution.evaluator import Evaluator
from tg_model.execution.graph_compiler import compile_graph
from tg_model.execution.run_context import RunContext
from tg_model.execution.validation import validate_graph

reset_commercial_aircraft_types()
_cm_adv = CargoJetProgram.instantiate()
_ac_adv = _cm_adv.aircraft
_in = nominal_inputs(_cm_adv, _ac_adv)
_graph, _handlers = compile_graph(_cm_adv)
assert validate_graph(_graph, configured_model=_cm_adv).passed
_by_id = {slot.stable_id: val for slot, val in _in.items()}
_res = Evaluator(_graph, compute_handlers=_handlers).evaluate(RunContext(), inputs=_by_id)
print("Explicit pipeline passed:", _res.passed, "(should match nominal case above)")


Explicit pipeline passed: True (should match nominal case above)


## Stress case (desk fails; envelope may still pass)

Lower **`mission_desk_baseline_max_range_m`** to **6,000 km** while keeping an **8,000 km** requested range. The **desk margin** goes negative, so `mission_range_margin_non_negative` **fails**. The **envelope** constraints can still **pass**, which shows the two tracks are **not** the same check.


In [5]:
def stress_desk_inputs(cm, ac):
    out = nominal_inputs(cm, ac)
    out[cm.mission_desk_baseline_max_range_m] = Quantity(6_000_000, m)
    return out


run_case("stressed desk baseline (expect FAIL on desk track)", stress_desk_inputs)



CASE: stressed desk baseline (expect FAIL on desk track)
Atlas-400F (notional) — evaluation snapshot
Verdict (desk margin + declared-envelope constraints + other checks): FAIL
Thesis (read this once)
The demo uses two independent ideas: (1) **Mission desk** — a toy
external scaling model produces `mission_range_margin_m` and
`mission_range_margin_non_negative` checks it. (2) **Declared envelope**
— scenario payload/range are compared to rolled-up masses and parameters
(`notional_takeoff_mass_closure`, payload/range vs envelope). Passing
both is not a single physics closure; it is stitched verification for
teaching.
Mission desk track:
  Margin (raw): -1953657.9860698776 m
  Margin (rounded): ~-1,953.7 km
  Margin ≥ 0: False
Declared envelope track (maximum takeoff weight / payload / range vs parameters):
  Envelope constraints all passed: True
  Evaluator completed without engine failures: False

Scenario inputs
quantity                          | value     | human (length)          
